In [2]:
import pandas as pd                                   # importamos Pandas con el apodo pd

df = pd.read_csv("../data/imdb_top_1000.csv")         # leemos el CSV que está en la carpeta data
print(df.shape)                                       # muestra (filas, columnas)
df.head()                                             # muestra las primeras 5 filas

(1000, 10)


,Unnamed: 0,Title,Certificate,Duration,Genre,Rate,Metascore,Description,Cast,Info
0,0,1. The Shawshank Redemption (1994),R,142 min,Drama,9.3,80.0,Two imprisoned men bond over a number of years...,"Director: Frank Darabont | Stars: Tim Robbins,...","Votes: 2,295,987 | Gross: $28.34M"
1,1,2. The Godfather (1972),R,175 min,"Crime, Drama",9.2,100.0,The aging patriarch of an organized crime dyna...,Director: Francis Ford Coppola | Stars: Marlon...,"Votes: 1,584,782 | Gross: $134.97M"
2,2,3. The Dark Knight (2008),PG-13,152 min,"Action, Crime, Drama",9.0,84.0,When the menace known as the Joker wreaks havo...,Director: Christopher Nolan | Stars: Christian...,"Votes: 2,260,649 | Gross: $534.86M"
3,3,4. The Godfather: Part II (1974),R,202 min,"Crime, Drama",9.0,90.0,The early life and career of Vito Corleone in ...,Director: Francis Ford Coppola | Stars: Al Pac...,"Votes: 1,107,253 | Gross: $57.30M"
4,4,5. The Lord of the Rings: The Return of the Ki...,PG-13,201 min,"Action, Adventure, Drama",8.9,94.0,Gandalf and Aragorn lead the World of Men agai...,"Director: Peter Jackson | Stars: Elijah Wood, ...","Votes: 1,614,369 | Gross: $377.85M"


In [3]:
df = df.drop(columns=["Unnamed: 0"])                  # borramos la columna "Unnamed: 0" (solo es un contador viejo)
print("Filas duplicadas:", df.duplicated().sum())     # cuenta cuántas filas son copia exacta de otra
df = df.drop_duplicates().reset_index(drop=True)      # borra las copias y vuelve a numerar las filas desde 0
print(df.shape)                                       # muestra el nuevo tamaño

Filas duplicadas: 602
(398, 9)


In [4]:
# ---------- Columnas nuevas a partir de las originales ----------
df["Año"] = df["Title"].str.extract(r"\((\d{4})\)")[0].astype(int)              # saca el año de "(1994)"
df["Titulo"] = (df["Title"]
                .str.replace(r"^\d+\.\s*", "", regex=True)                       # quita el "1. " del inicio
                .str.replace(r"\s*\(\d{4}\)$", "", regex=True))                  # quita el " (1994)" del final
df["Duracion"] = df["Duration"].str.replace(" min", "").astype(int)              # "142 min" → 142
df["Genero"] = df["Genre"].str.split(", ").str[0]                                # "Crime, Drama" → "Crime" (el principal)
df["Director"] = df["Cast"].str.extract(r"Directors?:\s*([^|]+)")[0].str.strip() # saca el nombre después de "Director:"
df["Votos"] = (df["Info"].str.extract(r"Votes:\s*([\d,]+)")[0]                   # saca "2,295,987"
               .str.replace(",", "").astype(int))                               # quita las comas → 2295987
df["Recaudacion"] = df["Info"].str.extract(r"Gross:\s*\$([\d.]+)M")[0].astype(float)   # "$28.34M" → 28.34 (millones)
df["Clasificacion"] = df["Certificate"].fillna("Sin dato")                       # rellena las clasificaciones vacías
df["Popularidad"] = pd.qcut(df["Votos"], 3, labels=["Baja", "Media", "Alta"])    # divide los votos en 3 grupos iguales
df["Codigo"] = ["PEL-" + str(i + 1).zfill(3) for i in range(len(df))]            # PEL-001, PEL-002, …

# ---------- Nombres en español y orden final ----------
df = df.rename(columns={"Rate": "Nota", "Genre": "Generos", "Description": "Descripcion"})   # traduce nombres
columnas = ["Codigo", "Titulo", "Genero", "Nota", "Duracion", "Director", "Popularidad",       # las 6 de la guía
            "Año", "Clasificacion", "Metascore", "Recaudacion", "Votos",                      # las adicionales
            "Generos", "Descripcion"]                                                         # extras útiles
df = df[columnas]                                                                # deja solo esas columnas, en ese orden

# ---------- Revisar y guardar ----------
print(df.shape)                                                                  # tamaño final
print(df.isna().sum())                                                           # cuántos vacíos hay por columna
df.to_csv("../data/peliculas_limpio.csv", index=False)                           # guarda el CSV limpio
df.head()                                                                        # muestra las primeras filas

(398, 14)
Codigo             0
Titulo             0
Genero             0
Nota               0
Duracion           0
Director           0
Popularidad        0
Año                0
Clasificacion      0
Metascore        112
Recaudacion       66
Votos              0
Generos            0
Descripcion        0
dtype: int64


,Codigo,Titulo,Genero,Nota,Duracion,Director,Popularidad,Año,Clasificacion,Metascore,Recaudacion,Votos,Generos,Descripcion
0,PEL-001,The Shawshank Redemption,Drama,9.3,142,Frank Darabont,Alta,1994,R,80.0,28.34,2295987,Drama,Two imprisoned men bond over a number of years...
1,PEL-002,The Godfather,Crime,9.2,175,Francis Ford Coppola,Alta,1972,R,100.0,134.97,1584782,"Crime, Drama",The aging patriarch of an organized crime dyna...
2,PEL-003,The Dark Knight,Action,9.0,152,Christopher Nolan,Alta,2008,PG-13,84.0,534.86,2260649,"Action, Crime, Drama",When the menace known as the Joker wreaks havo...
3,PEL-004,The Godfather: Part II,Crime,9.0,202,Francis Ford Coppola,Alta,1974,R,90.0,57.30,1107253,"Crime, Drama",The early life and career of Vito Corleone in ...
4,PEL-005,The Lord of the Rings: The Return of the King,Action,8.9,201,Peter Jackson,Alta,2003,PG-13,94.0,377.85,1614369,"Action, Adventure, Drama",Gandalf and Aragorn lead the World of Men agai...
